# 06 神经网络完整流程

每节下方的完整代码块均可单独复制运行，包含自己的 import、数据加载和实际调用。使用全部订单，不做行采样。安装所需依赖后，从 `examples` 目录运行。

In [ ]:
import sys
sys.path.append('../')

## 1. ResNet 分类：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import ResNet, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "MOB6@30"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = ResNet(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = ResNet(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["auc", "ks", "brier"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = ResNet(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric=get_metric("brier"),
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["auc", "ks", "brier"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_ResNet_binary_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = ResNet.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.5325
1,1,预训练验证集1,validation_0,1,重建损失,1.3567


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.1783
1,1,训练集,train,2,损失,0.1434
2,1,验证集,validation_0,1,损失,0.1560
3,1,验证集,validation_0,2,损失,0.1284


,AUC,KS,Brier
0,0.5054,0.0275,0.2232


[I 2026-11-09 14:02:17,719] A new study created in memory with name: no-name-52c7709b-0b6b-44d3-8a5a-ccbf46d99820


[I 2026-11-09 14:02:18,048] Trial 0 finished with value: {'概率均方误差': 0.12781208637109495} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 0.12781208637109495.


[I 2026-11-09 14:02:18,385] Trial 1 finished with value: {'概率均方误差': 0.22263231224613925} and parameters: {'learning_rate': 0.001}. Best is trial 0 with value: 0.12781208637109495.


,number,value_概率均方误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state,LIFT@1%,LIFT@3%,LIFT@5%,LIFT@10%
0,0,0.1278,2026-11-09 14:02:17.722252,2026-11-09 14:02:18.048269,0 days 00:00:00.326017,0.0030,[{'概率均方误差': 0.12781208637109495}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE,1.4086,1.3410,1.3150,1.2938
1,1,0.2226,2026-11-09 14:02:18.049412,2026-11-09 14:02:18.385680,0 days 00:00:00.336268,0.0010,[{'概率均方误差': 0.22263231224613925}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE,0.6339,0.8234,0.9474,1.0534


统计指标        KS                         AUC                         样本数                                    坏样本率                     
数据集        训练集    验证集    测试集    OOT    训练集    验证集    测试集    OOT    训练集        验证集        测试集        OOT    训练集    验证集    测试集    OOT
逾期指标                                                                                                                               
MOB6@30 0.0629 0.0552 0.0510 0.0647 0.5445 0.5342 0.5298 0.5383  46881 15635.0000 20838.0000 28025.0000 0.0911 0.0904 0.0934 0.0894

,预测结果
78333,0
9091,0
106130,0
61444,0
30329,0


C:\Users\18306\AppData\Local\Temp\hscredit_ResNet_binary_92jvwvfl


## 2. ResNet 回归：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import ResNet, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "风险调整金额"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = ResNet(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = ResNet(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["rmse", "mae", "r2"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = ResNet(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric="rmse",
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["rmse", "mae", "r2"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_ResNet_regression_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = ResNet.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.5325
1,1,预训练验证集1,validation_0,1,重建损失,1.3567


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.3520
1,1,训练集,train,2,损失,0.3336
2,1,验证集,validation_0,1,损失,0.3339
3,1,验证集,validation_0,2,损失,0.3308


,均方根误差,平均绝对误差,决定系数
0,4156.8111,2612.5960,0.0150


[I 2026-11-09 14:02:55,193] A new study created in memory with name: no-name-e008a8b5-8144-488a-82fa-d086f9db11dd


[I 2026-11-09 14:02:55,420] Trial 0 finished with value: {'均方根误差': 4112.970720590446} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 4112.970720590446.


[I 2026-11-09 14:02:55,651] Trial 1 finished with value: {'均方根误差': 4122.871910196234} and parameters: {'learning_rate': 0.001}. Best is trial 0 with value: 4112.970720590446.


,number,value_均方根误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state
0,0,4112.9707,2026-11-09 14:02:55.196569,2026-11-09 14:02:55.420123,0 days 00:00:00.223554,0.0030,[{'均方根误差': 4112.970720590446}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE
1,1,4122.8719,2026-11-09 14:02:55.421295,2026-11-09 14:02:55.651162,0 days 00:00:00.229867,0.0010,[{'均方根误差': 4122.871910196234}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE


统计指标          样本数                                      均方根误差                                  平均绝对误差                                 决定系数                      
数据集           训练集        验证集        测试集        OOT       训练集       验证集       测试集       OOT       训练集       验证集       测试集       OOT    训练集    验证集    测试集     OOT
目标变量                                                                                                                                                           
风险调整金额 46881.0000 15635.0000 20838.0000 28025.0000 4115.9693 4112.9707 4137.8509 5445.5850 2584.4280 2588.5238 2595.9454 3879.7370 0.0311 0.0263 0.0240 -0.0001

,预测结果
78333,4694.0248
9091,4347.9866
106130,4499.2041
61444,4193.9353
30329,4534.7254


C:\Users\18306\AppData\Local\Temp\hscredit_ResNet_regression_l8xt15jq


## 3. FTTransformer 分类：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import FTTransformer, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "MOB6@30"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = FTTransformer(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    n_heads=4,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = FTTransformer(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    n_heads=4,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["auc", "ks", "brier"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = FTTransformer(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    n_heads=4,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric=get_metric("brier"),
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["auc", "ks", "brier"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_FTTransformer_binary_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = FTTransformer.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.5590
1,1,预训练验证集1,validation_0,1,重建损失,1.4651


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.1406
1,1,训练集,train,2,损失,0.1225
2,1,验证集,validation_0,1,损失,0.1311
3,1,验证集,validation_0,2,损失,0.1152


,AUC,KS,Brier
0,0.5220,0.0446,0.2000


[I 2026-11-09 14:03:25,538] A new study created in memory with name: no-name-97b711ce-9cb2-489c-b063-4319045f5f22


[I 2026-11-09 14:03:26,998] Trial 0 finished with value: {'概率均方误差': 0.10655029451617107} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 0.10655029451617107.


[I 2026-11-09 14:03:28,471] Trial 1 finished with value: {'概率均方误差': 0.10461841769333528} and parameters: {'learning_rate': 0.001}. Best is trial 1 with value: 0.10461841769333528.


,number,value_概率均方误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state,LIFT@1%,LIFT@3%,LIFT@5%,LIFT@10%
0,0,0.1066,2026-11-09 14:03:25.541077,2026-11-09 14:03:26.998200,0 days 00:00:01.457123,0.0030,[{'概率均方误差': 0.10655029451617107}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE,1.4790,1.4116,1.1595,1.1948
1,1,0.1046,2026-11-09 14:03:26.999868,2026-11-09 14:03:28.471835,0 days 00:00:01.471967,0.0010,[{'概率均方误差': 0.10461841769333528}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE,1.1973,1.2704,1.3716,1.3928


统计指标        KS                         AUC                         样本数                                    坏样本率                     
数据集        训练集    验证集    测试集    OOT    训练集    验证集    测试集    OOT    训练集        验证集        测试集        OOT    训练集    验证集    测试集    OOT
逾期指标                                                                                                                               
MOB6@30 0.0890 0.0942 0.0917 0.0916 0.5574 0.5605 0.5576 0.5601  46881 15635.0000 20838.0000 28025.0000 0.0911 0.0904 0.0934 0.0894

,预测结果
78333,0
9091,0
106130,0
61444,0
30329,0


C:\Users\18306\AppData\Local\Temp\hscredit_FTTransformer_binary_b3hieqvp


## 4. FTTransformer 回归：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import FTTransformer, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "风险调整金额"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = FTTransformer(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    n_heads=4,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = FTTransformer(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    n_heads=4,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["rmse", "mae", "r2"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = FTTransformer(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    n_heads=4,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric="rmse",
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["rmse", "mae", "r2"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_FTTransformer_regression_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = FTTransformer.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.5590
1,1,预训练验证集1,validation_0,1,重建损失,1.4651


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.3459
1,1,训练集,train,2,损失,0.3431
2,1,验证集,validation_0,1,损失,0.3433
3,1,验证集,validation_0,2,损失,0.3409


,均方根误差,平均绝对误差,决定系数
0,4203.0331,2669.0140,-0.0070


[I 2026-11-09 14:04:07,356] A new study created in memory with name: no-name-7b996258-9e53-4cc3-8859-4cf2894ae6c1


[I 2026-11-09 14:04:08,508] Trial 0 finished with value: {'均方根误差': 4129.775248453976} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 4129.775248453976.


[I 2026-11-09 14:04:09,674] Trial 1 finished with value: {'均方根误差': 4137.1542677996495} and parameters: {'learning_rate': 0.001}. Best is trial 0 with value: 4129.775248453976.


,number,value_均方根误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state
0,0,4129.7752,2026-11-09 14:04:07.359893,2026-11-09 14:04:08.508373,0 days 00:00:01.148480,0.0030,[{'均方根误差': 4129.775248453976}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE
1,1,4137.1543,2026-11-09 14:04:08.510080,2026-11-09 14:04:09.674152,0 days 00:00:01.164072,0.0010,[{'均方根误差': 4137.1542677996495}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE


统计指标          样本数                                      均方根误差                                  平均绝对误差                                 决定系数                     
数据集           训练集        验证集        测试集        OOT       训练集       验证集       测试集       OOT       训练集       验证集       测试集       OOT    训练集    验证集    测试集    OOT
目标变量                                                                                                                                                          
风险调整金额 46881.0000 15635.0000 20838.0000 28025.0000 4135.6255 4129.7752 4147.3956 5408.7431 2576.2171 2576.8089 2582.1637 3802.6033 0.0218 0.0183 0.0195 0.0133

,预测结果
78333,4416.8130
9091,4605.6757
106130,4659.0103
61444,4455.6261
30329,4615.0159


C:\Users\18306\AppData\Local\Temp\hscredit_FTTransformer_regression_zz3ysm46


## 5. TabM 分类：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import TabM, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "MOB6@30"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = TabM(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    k=2,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = TabM(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    k=2,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["auc", "ks", "brier"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = TabM(
    target=target,
    objective="binary",
    d_block=16,
    n_blocks=1,
    k=2,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric=get_metric("brier"),
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["auc", "ks", "brier"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_TabM_binary_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = TabM.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.3774
1,1,预训练验证集1,validation_0,1,重建损失,1.2634


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.1337
1,1,训练集,train,2,损失,0.1012
2,1,验证集,validation_0,1,损失,0.1108
3,1,验证集,validation_0,2,损失,0.0876


,AUC,KS,Brier
0,0.5192,0.0348,0.1372


[I 2026-11-09 14:04:37,435] A new study created in memory with name: no-name-1ee7293c-1dd6-4179-8b4a-976bfc6694d8


[I 2026-11-09 14:04:37,790] Trial 0 finished with value: {'概率均方误差': 0.10924813354026149} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 0.10924813354026149.


[I 2026-11-09 14:04:38,140] Trial 1 finished with value: {'概率均方误差': 0.19104700844722872} and parameters: {'learning_rate': 0.001}. Best is trial 0 with value: 0.10924813354026149.


,number,value_概率均方误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state,LIFT@1%,LIFT@3%,LIFT@5%,LIFT@10%
0,0,0.1092,2026-11-09 14:04:37.438728,2026-11-09 14:04:37.790019,0 days 00:00:00.351291,0.0030,[{'概率均方误差': 0.10924813354026149}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE,1.6199,1.3880,1.3009,1.2867
1,1,0.1910,2026-11-09 14:04:37.791462,2026-11-09 14:04:38.140417,0 days 00:00:00.348955,0.0010,[{'概率均方误差': 0.19104700844722872}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE,1.0564,1.1293,1.2302,1.1170


统计指标        KS                         AUC                         样本数                                    坏样本率                     
数据集        训练集    验证集    测试集    OOT    训练集    验证集    测试集    OOT    训练集        验证集        测试集        OOT    训练集    验证集    测试集    OOT
逾期指标                                                                                                                               
MOB6@30 0.0540 0.0737 0.0522 0.0587 0.5360 0.5440 0.5331 0.5423  46881 15635.0000 20838.0000 28025.0000 0.0911 0.0904 0.0934 0.0894

,预测结果
78333,0
9091,0
106130,0
61444,0
30329,0


C:\Users\18306\AppData\Local\Temp\hscredit_TabM_binary_g730813d


## 6. TabM 回归：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import TabM, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "风险调整金额"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = TabM(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    k=2,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = TabM(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    k=2,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["rmse", "mae", "r2"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = TabM(
    target=target,
    objective="regression",
    d_block=16,
    n_blocks=1,
    k=2,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric="rmse",
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["rmse", "mae", "r2"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_TabM_regression_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = TabM.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.3774
1,1,预训练验证集1,validation_0,1,重建损失,1.2634


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.3415
1,1,训练集,train,2,损失,0.3310
2,1,验证集,validation_0,1,损失,0.3304
3,1,验证集,validation_0,2,损失,0.3283


,均方根误差,平均绝对误差,决定系数
0,4147.5797,2595.1029,0.0194


[I 2026-11-09 14:05:14,504] A new study created in memory with name: no-name-08685596-85b3-4a86-bbdf-b0900d3ad086


[I 2026-11-09 14:05:14,698] Trial 0 finished with value: {'均方根误差': 4126.272644187267} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 4126.272644187267.


[I 2026-11-09 14:05:14,887] Trial 1 finished with value: {'均方根误差': 4131.330242552284} and parameters: {'learning_rate': 0.001}. Best is trial 0 with value: 4126.272644187267.


,number,value_均方根误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state
0,0,4126.2726,2026-11-09 14:05:14.507804,2026-11-09 14:05:14.698910,0 days 00:00:00.191106,0.0030,[{'均方根误差': 4126.272644187267}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE
1,1,4131.3302,2026-11-09 14:05:14.699922,2026-11-09 14:05:14.887406,0 days 00:00:00.187484,0.0010,[{'均方根误差': 4131.330242552284}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE


统计指标          样本数                                      均方根误差                                  平均绝对误差                                 决定系数                     
数据集           训练集        验证集        测试集        OOT       训练集       验证集       测试集       OOT       训练集       验证集       测试集       OOT    训练集    验证集    测试集    OOT
目标变量                                                                                                                                                          
风险调整金额 46881.0000 15635.0000 20838.0000 28025.0000 4128.9876 4126.2726 4150.5823 5419.3501 2581.5941 2583.6398 2587.6415 3825.0336 0.0250 0.0200 0.0179 0.0095

,预测结果
78333,4460.7221
9091,4755.6378
106130,4739.7881
61444,4426.8665
30329,4596.2735


C:\Users\18306\AppData\Local\Temp\hscredit_TabM_regression_lbkztri0


## 7. TabNet 分类：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import TabNet, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "MOB6@30"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = TabNet(
    target=target,
    objective="binary",
    n_d=8,
    n_a=8,
    n_steps=2,
    virtual_batch_size=128,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = TabNet(
    target=target,
    objective="binary",
    n_d=8,
    n_a=8,
    n_steps=2,
    virtual_batch_size=128,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["auc", "ks", "brier"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = TabNet(
    target=target,
    objective="binary",
    n_d=8,
    n_a=8,
    n_steps=2,
    virtual_batch_size=128,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=FocalLoss(alpha=0.6, gamma=1.0),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric=get_metric("brier"),
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["auc", "ks", "brier"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_TabNet_binary_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = TabNet.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

C:\Users\18306\AppData\Roaming\Python\Python314\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.7748
1,1,预训练验证集1,validation_0,1,重建损失,0.9125


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.1273
1,1,训练集,train,2,损失,0.0855
2,1,验证集,validation_0,1,损失,0.0844
3,1,验证集,validation_0,2,损失,0.0832


,AUC,KS,Brier
0,0.5299,0.0546,0.1088


[I 2026-11-09 14:05:43,728] A new study created in memory with name: no-name-041a7d89-87ef-46a3-b17a-f3ac7118b9a2


[I 2026-11-09 14:05:45,504] Trial 0 finished with value: {'概率均方误差': 0.12437131665354867} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 0.12437131665354867.


[I 2026-11-09 14:05:47,999] Trial 1 finished with value: {'概率均方误差': 0.23864017075336458} and parameters: {'learning_rate': 0.001}. Best is trial 0 with value: 0.12437131665354867.


,number,value_概率均方误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state,LIFT@1%,LIFT@3%,LIFT@5%,LIFT@10%
0,0,0.1244,2026-11-09 14:05:43.730039,2026-11-09 14:05:45.504675,0 days 00:00:01.774636,0.0030,[{'概率均方误差': 0.12437131665354867}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE,0.8451,0.9410,1.0888,1.0605
1,1,0.2386,2026-11-09 14:05:45.505611,2026-11-09 14:05:47.999605,0 days 00:00:02.493994,0.0010,[{'概率均方误差': 0.23864017075336458}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE,0.4460,0.6352,0.7070,0.9332


统计指标        KS                         AUC                         样本数                                    坏样本率                     
数据集        训练集    验证集    测试集    OOT    训练集    验证集    测试集    OOT    训练集        验证集        测试集        OOT    训练集    验证集    测试集    OOT
逾期指标                                                                                                                               
MOB6@30 0.0308 0.0385 0.0219 0.0413 0.5064 0.5224 0.5005 0.5118  46881 15635.0000 20838.0000 28025.0000 0.0911 0.0904 0.0934 0.0894

,预测结果
78333,0
9091,0
106130,0
61444,0
30329,0


C:\Users\18306\AppData\Local\Temp\hscredit_TabNet_binary_l5uas3h1


## 8. TabNet 回归：预训练、训练、调参与部署

完整独立流程。使用全部样本；为了展示接口，预训练设为一轮、监督训练两轮，不将演示指标当作充分收敛的模型表现。

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from optuna.samplers import GridSampler
from hscredit import TabNet, ModelTuner, get_metric
from hscredit.core.models.losses import FocalLoss
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

features = features + ["GROUP", "BASINF_X19", "BASE_X61"]
target = "风险调整金额"

# 预训练只传特征，包含全部训练子集，没有标签和行采样。
pretrained = TabNet(
    target=target,
    objective="regression",
    n_d=8,
    n_a=8,
    n_steps=2,
    virtual_batch_size=128,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
pretrained.pretrain(
    fit_data[features], eval_set=[validation[features]], pretraining_ratio=0.4, max_epochs=1
)
display(pretrained.get_learning_curve())

# 从预训练快照开始监督训练，使用现成优化器、损失和评估指标。
model = TabNet(
    target=target,
    objective="regression",
    n_d=8,
    n_a=8,
    n_steps=2,
    virtual_batch_size=128,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    from_pretrained=pretrained,
)
display(model.get_learning_curve())
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["rmse", "mae", "r2"])]))

# 两个学习率候选均从相同预训练状态开始。
search_space = {"learning_rate": [0.001, 0.003]}
candidate = TabNet(
    target=target,
    objective="regression",
    n_d=8,
    n_a=8,
    n_steps=2,
    virtual_batch_size=128,
    max_epochs=2,
    batch_size=2048,
    early_stopping_rounds=None,
    validation_fraction=0,
    device="auto",
    random_state=42,
    optimizer=torch.optim.AdamW,
    optimizer_params={"betas": (0.9, 0.98)},
    loss=torch.nn.SmoothL1Loss(reduction="none"),
)
tuner = ModelTuner(
    candidate,
    search_space=search_space,
    fit_params={"from_pretrained": pretrained},
    metric="rmse",
    cv=None,
    n_jobs=1,
    random_state=42,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())
display(tuner.plot_optimization_history())
best_model = tuner.get_best_model()

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    best_model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["rmse", "mae", "r2"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_TabNet_regression_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
pretrained.save(output / "预训练模型.pkl")
best_model.save(output / "完整模型.pkl")
best_model.save_inference(output / "推理模型.pkl")

restored = TabNet.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["预测结果"]).head()
)
print(output)

C:\Users\18306\AppData\Roaming\Python\Python314\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,重建损失,1.7748
1,1,预训练验证集1,validation_0,1,重建损失,0.9125


,训练修订,数据集,原生数据集,迭代,指标,数值
0,1,训练集,train,1,损失,0.4107
1,1,训练集,train,2,损失,0.3339
2,1,验证集,validation_0,1,损失,0.3381
3,1,验证集,validation_0,2,损失,0.3295


,均方根误差,平均绝对误差,决定系数
0,4160.7736,2608.1334,0.0131


[I 2026-11-09 14:06:34,416] A new study created in memory with name: no-name-b33b11a2-3560-491c-bb23-575b1654c9db


[I 2026-11-09 14:06:36,037] Trial 0 finished with value: {'均方根误差': 4244.385810025311} and parameters: {'learning_rate': 0.003}. Best is trial 0 with value: 4244.385810025311.


[I 2026-11-09 14:06:37,782] Trial 1 finished with value: {'均方根误差': 5082.737397643596} and parameters: {'learning_rate': 0.001}. Best is trial 0 with value: 4244.385810025311.


,number,value_均方根误差,datetime_start,datetime_complete,duration,params_learning_rate,user_attrs_各折指标,system_attrs_grid_id,system_attrs_search_space,state
0,0,4244.3858,2026-11-09 14:06:34.416992,2026-11-09 14:06:36.037008,0 days 00:00:01.620016,0.0030,[{'均方根误差': 4244.385810025311}],0,"{'learning_rate': [0.001, 0.003]}",COMPLETE
1,1,5082.7374,2026-11-09 14:06:36.038092,2026-11-09 14:06:37.782771,0 days 00:00:01.744679,0.0010,[{'均方根误差': 5082.737397643596}],1,"{'learning_rate': [0.001, 0.003]}",COMPLETE


统计指标          样本数                                      均方根误差                                  平均绝对误差                                  决定系数                        
数据集           训练集        验证集        测试集        OOT       训练集       验证集       测试集       OOT       训练集       验证集       测试集       OOT     训练集     验证集     测试集     OOT
目标变量                                                                                                                                                              
风险调整金额 46881.0000 15635.0000 20838.0000 28025.0000 4222.3943 4244.3858 4227.4499 5550.1266 2686.8527 2703.7493 2689.1900 3949.7256 -0.0196 -0.0369 -0.0188 -0.0389

,预测结果
78333,4381.3934
9091,4239.0747
106130,5113.0064
61444,4082.5562
30329,4986.8535


C:\Users\18306\AppData\Local\Temp\hscredit_TabNet_regression_9j472g1q


## 9. TabPFN 分类：全量预训练、微调与报告

本节同样不采样。超过官方预训练样本规模时，关闭的是规模检查，不是资源需求；全量上下文与微调需要充足内存和合适的 GPU。当前 32 GiB CPU 环境不执行此重型段落，不能把此前小样本验证视为全量验证。

In [ ]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from optuna.samplers import GridSampler
from hscredit import TabPFN, ModelTuner, get_metric
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

target = "MOB6@30"
official = TabPFNClassifier.create_default_for_version(
    ModelVersion.V2, n_estimators=1, device="auto"
)
model = TabPFN(
    target=target,
    objective="binary",
    model_path=official.model_path,
    n_estimators=1,
    device="auto",
    n_jobs=1,
    random_state=42,
    inference_precision=torch.float32,
    ignore_pretraining_limits=True,
    memory_saving_mode=True,
    finetune_params={
        "model_version": "v2",
        "max_epochs": 1,
        "batch_size": 2048,
        "validation_fraction": 0,
        "early_stopping_rounds": None,
        "use_lr_scheduler": False,
    },
)

# 全量 X-only 字段预测预训练，然后分别展示上下文拟合和官方微调。
model.pretrain(fit_data[features], pretraining_ratio=0.4, max_epochs=1)
display(model.get_learning_curve())
model.fit(fit_data[features + [target]], finetune=False)
display(
    pd.DataFrame(model.predict(test[features]), index=test.index, columns=["上下文预测"]).head()
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    finetune=True,
)
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["auc", "ks", "brier"])]))

# 调参控制推理温度；不向官方后端塞入其他网络的自定义损失接口。
search_space = {"softmax_temperature": [0.8, 1.0]}
tuner = ModelTuner(
    model,
    search_space=search_space,
    metric=get_metric("brier"),
    cv=None,
    n_jobs=1,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["auc", "ks", "brier"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_tabpfn_binary_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
model.save_inference(output / "推理模型.pkl")
restored = TabPFN.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["恢复后预测"]).head()
)
print(output)

## 10. TabPFN 回归：全量预训练、微调与报告

本节同样不采样。超过官方预训练样本规模时，关闭的是规模检查，不是资源需求；全量上下文与微调需要充足内存和合适的 GPU。当前 32 GiB CPU 环境不执行此重型段落，不能把此前小样本验证视为全量验证。

In [ ]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd
from IPython.display import display
from hscredit import hscredit_dataset
from hscredit.utils.datasets import HSCREDIT_EXAMPLE_COLUMNS
from pathlib import Path
from tempfile import mkdtemp
import torch
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion
from optuna.samplers import GridSampler
from hscredit import TabPFN, ModelTuner, get_metric
from hscredit.report import ModelReport

from hscredit.core.model_selection import oot_split, group_train_test_split

data = hscredit_dataset(columns=HSCREDIT_EXAMPLE_COLUMNS)

features = ["BASE_X187", "DK_X1689", "DK_X44", "DK_X1995_2", "DK_X1223", "DK_X2677"]
target = "MOB6@30"
data["MOB1@30"] = data["MOB1"].gt(30).astype(int)
data["MOB3@30"] = data["MOB3"].gt(30).astype(int)
data["MOB6@30"] = data["MOB6"].gt(30).astype(int)
data["风险调整金额"] = data["loan_amt"].where(data["MOB6"].le(60), -data["loan_amt"])
data["授信时间"] = pd.to_datetime(data["授信时间"])

# 最新自然月作为 OOT；同一客户不能同时进入开发集和 OOT。
oot_start = data["授信时间"].max().to_period("M").start_time
development, oot = oot_split(data, time_col="授信时间", oot_start=oot_start)
development = development.loc[~development["用户编码"].isin(oot["用户编码"])].copy()
train, test = group_train_test_split(development, group="用户编码", test_size=0.25, random_state=42)
fit_data, validation = group_train_test_split(
    train, group="用户编码", test_size=0.25, random_state=42
)

target = "风险调整金额"
official = TabPFNRegressor.create_default_for_version(
    ModelVersion.V2, n_estimators=1, device="auto"
)
model = TabPFN(
    target=target,
    objective="regression",
    model_path=official.model_path,
    n_estimators=1,
    device="auto",
    n_jobs=1,
    random_state=42,
    inference_precision=torch.float32,
    ignore_pretraining_limits=True,
    memory_saving_mode=True,
    finetune_params={
        "model_version": "v2",
        "max_epochs": 1,
        "batch_size": 2048,
        "validation_fraction": 0,
        "early_stopping_rounds": None,
        "use_lr_scheduler": False,
    },
)

# 全量 X-only 字段预测预训练，然后分别展示上下文拟合和官方微调。
model.pretrain(fit_data[features], pretraining_ratio=0.4, max_epochs=1)
display(model.get_learning_curve())
model.fit(fit_data[features + [target]], finetune=False)
display(
    pd.DataFrame(model.predict(test[features]), index=test.index, columns=["上下文预测"]).head()
)
model.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    finetune=True,
)
display(pd.DataFrame([model.evaluate(test[features + [target]], metrics=["rmse", "mae", "r2"])]))

# 调参控制推理温度；不向官方后端塞入其他网络的自定义损失接口。
search_space = {"softmax_temperature": [0.8, 1.0]}
tuner = ModelTuner(
    model,
    search_space=search_space,
    metric="rmse",
    cv=None,
    n_jobs=1,
    sampler=GridSampler(search_space, seed=42),
    retention="predictions",
)
tuner.fit(
    fit_data[features + [target]],
    eval_set={"验证集": validation[features + [target]]},
    n_trials=2,
    show_progress_bar=False,
)
display(tuner.get_optimization_history())

datasets = {"训练集": fit_data, "验证集": validation, "测试集": test, "OOT": oot}
report = ModelReport(
    model,
    datasets=datasets,
    features=features,
    target=target,
    metrics=["rmse", "mae", "r2"],
    n_jobs=1,
)
display(report.summary())
output = Path(mkdtemp(prefix="hscredit_tabpfn_regression_"))
report.to_excel(
    output / "模型报告.xlsx", with_plots=False, include_sample_records=False, mode="strict"
)
model.save_inference(output / "推理模型.pkl")
restored = TabPFN.load(output / "推理模型.pkl")
display(
    pd.DataFrame(restored.predict(test[features]), index=test.index, columns=["恢复后预测"]).head()
)
print(output)